# Qwen-3B Full-Stack Pipeline (Colab)

This is the condensed runbook for the Qwen-3B path-to-100+ TPS stack.

Run the cells top to bottom. Each stage is resumable and skips artifacts that already exist on Drive.

| Cell | Purpose |
|---|---|
| 1 | Mount Drive, install deps, pull fresh `main`, set all run knobs. |
| 2 | Build/resume corpus, compute AWQ smoothing, build frozen baseline. |
| 3 | Train Eagle5 candidates and run tau@4 ranking. |
| 4 | Search frontier policies: variable-K, entropy routing, lattice probe, AWQ/W4A8 validation. |
| 5 | Write `summary.md` with winner, projected TPS, and local bench command. |

Default mode is aggressive but bounded for Colab Pro: `MAX_TPS_MODE=True`, 10k corpus target, a wider LR grid, three residual-delta variants, and frontier search only on the top tau candidates.


In [ ]:
# Cell 1 - Setup, deps, repo, config
from google.colab import drive
drive.mount('/content/drive')

import glob, json, math, os, shutil, subprocess, sys, time
from pathlib import Path


def run(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)

run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets; Hub downloads get higher limits.')
else:
    print('No HF_TOKEN secret found; public Hub downloads still work.')

import numpy as np
import pyarrow
import torch
import transformers

assert torch.cuda.is_available(), 'No CUDA: set Runtime -> Change runtime type -> GPU'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')
print(f'transformers={transformers.__version__} torch={torch.__version__} pyarrow={pyarrow.__version__}')

# Main knobs. Change these before running downstream cells.
MAX_TPS_MODE = True
BUILD_CORPUS_IF_NEEDED = True     # Set False if another Colab is actively writing the shared corpus.
FORCE_RETRAIN = False
FORCE_TAU = False
FORCE_FRONTIER = False
FORCE_VALIDATE = False
USE_TORCH_COMPILE = False         # Safer default on Colab Pro; turn on only after a clean run.

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
CAPTURE_LAYER = 32
BASELINE_DEC_TPS = 26.6
W4A8_MULTIPLIER = 1.25
SPEC_EFFICIENCY = 0.85

DRIVE_ROOT = '/content/drive/MyDrive/dismantle'
ARTIFACT_DIR = f'{DRIVE_ROOT}/qwen3b_artifacts'
CORPUS_DIR = f'{DRIVE_ROOT}/qwen3b_corpus'
LOCAL_CORPUS = '/content/qwen3b_corpus'

CORPUS_TARGET_SEQS = 10_000 if MAX_TPS_MODE else 2_000
TRAIN_MAX_ROWS = 8_000 if MAX_TPS_MODE else 4_000
TRAIN_EPOCHS = 10 if MAX_TPS_MODE else 8
TRAIN_MAX_ROW_TOKENS = 192 if MAX_TPS_MODE else 128
EVAL_MAX_WINDOWS = 6_000 if MAX_TPS_MODE else 2_000
FRONTIER_MAX_DEPTH = 12 if MAX_TPS_MODE else 6
FRONTIER_TOP_N = 3 if MAX_TPS_MODE else 1
LRs = ([1e-4, 2e-4, 3e-4, 6e-4, 1e-3, 2e-3, 3e-3]
       if MAX_TPS_MODE else [3e-4, 1e-3, 3e-3])

TRAIN_SPECS = [{'lr': lr, 'residual_delta': 0.0, 'calib_weight': 0.10, 'family': 'base'} for lr in LRs]
if MAX_TPS_MODE:
    TRAIN_SPECS += [
        {'lr': 3e-4, 'residual_delta': 0.01, 'calib_weight': 0.15, 'family': 'rd'},
        {'lr': 6e-4, 'residual_delta': 0.02, 'calib_weight': 0.20, 'family': 'rd'},
        {'lr': 1e-3, 'residual_delta': 0.02, 'calib_weight': 0.20, 'family': 'rd'},
    ]

for d in (DRIVE_ROOT, ARTIFACT_DIR, CORPUS_DIR, LOCAL_CORPUS):
    os.makedirs(d, exist_ok=True)

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/dismantle'])
else:
    run(['git', '-C', '/content/dismantle', 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', '/content/dismantle', 'checkout', BRANCH])
    run(['git', '-C', '/content/dismantle', 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir('/content/dismantle')
print('repo:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

REQUIRED_FILES = [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/awq_w4a8_validate.py',
    'colab/build_qwen3b_frozen_hf.py',
    'tools/training/awq_calibrate.py',
]
for path in REQUIRED_FILES:
    assert os.path.exists(path), f'missing required file: {path}'
    print('ok', path)

print(json.dumps({
    'MAX_TPS_MODE': MAX_TPS_MODE,
    'CORPUS_TARGET_SEQS': CORPUS_TARGET_SEQS,
    'TRAIN_MAX_ROWS': TRAIN_MAX_ROWS,
    'TRAIN_EPOCHS': TRAIN_EPOCHS,
    'TRAIN_MAX_ROW_TOKENS': TRAIN_MAX_ROW_TOKENS,
    'EVAL_MAX_WINDOWS': EVAL_MAX_WINDOWS,
    'FRONTIER_MAX_DEPTH': FRONTIER_MAX_DEPTH,
    'FRONTIER_TOP_N': FRONTIER_TOP_N,
    'TRAIN_SPECS': TRAIN_SPECS,
    'ARTIFACT_DIR': ARTIFACT_DIR,
    'CORPUS_DIR': CORPUS_DIR,
}, indent=2))


In [ ]:
# Cell 2 - Prepare corpus, AWQ, frozen baseline
STATS_FILE = f'{CORPUS_DIR}/per_site_activation_stats.npz'
MANIFEST = f'{CORPUS_DIR}/manifest.json'
AWQ_OUT = f'{ARTIFACT_DIR}/qwen3b_awq_smoothing.json'
FROZEN_OUT = f'{ARTIFACT_DIR}/qwen3b_frozen.npz'


def stats_sequences(path):
    if not os.path.exists(path):
        return 0
    try:
        with np.load(path) as z:
            return int(np.asarray(z['sequences_written']).item()) if 'sequences_written' in z.files else 0
    except Exception as e:
        print(f'WARN: could not read stats file {path}: {e}')
        return 0


def corpus_state():
    shards = len(glob.glob(f'{CORPUS_DIR}/shard_*.parquet'))
    seqs = stats_sequences(STATS_FILE)
    manifest_target = 0
    if os.path.exists(MANIFEST):
        try:
            manifest_target = int(json.load(open(MANIFEST)).get('max_sequences', 0))
        except Exception as e:
            print(f'WARN: could not read manifest: {e}')
    required_shards = math.ceil(CORPUS_TARGET_SEQS / 16)
    complete = shards >= required_shards and seqs >= CORPUS_TARGET_SEQS
    return {'shards': shards, 'seqs': seqs, 'required_shards': required_shards, 'manifest_target': manifest_target, 'complete': complete}

state = corpus_state()
print('corpus state:', state)
if state['complete']:
    print('Corpus is complete enough for this run; skipping calibration build.')
elif not BUILD_CORPUS_IF_NEEDED:
    raise RuntimeError('Corpus incomplete and BUILD_CORPUS_IF_NEEDED=False. Wait for the other Colab or flip the knob in Cell 1.')
else:
    os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    if VRAM_GB >= 70:
        tier, load_4bit, batch, lm_head_chunk = 'G4/H100/Blackwell 70GB+', False, 8, 256
    elif VRAM_GB >= 35:
        tier, load_4bit, batch, lm_head_chunk = 'A100 40GB-class', False, 6, 128
    elif VRAM_GB >= 20:
        tier, load_4bit, batch, lm_head_chunk = 'L4 24GB-class', True, 4, 64
    else:
        tier, load_4bit, batch, lm_head_chunk = 'T4/V100 16GB-class', True, 2, 32
    print(f'Building/resuming corpus: tier={tier} batch={batch} chunk={lm_head_chunk} load={"4-bit" if load_4bit else "fp16"}')
    cmd = [
        sys.executable, 'colab/mega_calibrate.py',
        '--model', MODEL_ID,
        '--max-sequences', str(CORPUS_TARGET_SEQS),
        '--max-tokens', '2048',
        '--batch-size', str(batch),
        '--capture-layer', str(CAPTURE_LAYER),
        '--topk-logits', '100',
        '--lm-head-chunk-tokens', str(lm_head_chunk),
        '--shard-size', '16',
        '--sync-dir', CORPUS_DIR,
        '--sync-every', '4',
        '--delete-local-after-sync',
        '--out', LOCAL_CORPUS,
    ]
    if load_4bit:
        cmd.append('--load-4bit')
    run(cmd)
    print('corpus state after build:', corpus_state())

assert os.path.exists(STATS_FILE), f'missing activation stats: {STATS_FILE}'
run([
    sys.executable, 'tools/training/awq_calibrate.py',
    '--stats', STATS_FILE,
    '--out', AWQ_OUT,
    '--alpha', '0.5',
    '--model', MODEL_ID,
])
with open(AWQ_OUT) as f:
    awq = json.load(f)
print(f'AWQ factors={len(awq["smoothing_factors"])} sequences={awq.get("sequences_calibrated")} mean={awq["stats"]["mean_factor"]:.4f}')

run([
    sys.executable, 'colab/build_qwen3b_frozen_hf.py',
    '--model', MODEL_ID,
    '--out', FROZEN_OUT,
])
assert os.path.exists(FROZEN_OUT), f'missing frozen baseline: {FROZEN_OUT}'
print(f'frozen ready: {FROZEN_OUT} ({os.path.getsize(FROZEN_OUT)/1e9:.2f} GB)')


In [ ]:
# Cell 3 - Train Eagle5 candidates and run tau race
TRAIN_BATCH = 32 if VRAM_GB >= 35 else 16
TAU_DEPTH = 4


def spec_tag(spec):
    lr_tag = f"lr{spec['lr']:.0e}".replace('+', '').replace('-0', '-')
    if spec['family'] == 'base':
        return lr_tag
    rd = int(round(spec['residual_delta'] * 1000))
    cw = int(round(spec['calib_weight'] * 100))
    return f"{lr_tag}_{spec['family']}rd{rd:03d}_cw{cw:02d}"

trained = []
for spec in TRAIN_SPECS:
    tag = spec_tag(spec)
    ckpt_dir = f'{ARTIFACT_DIR}/checkpoints/eagle5_qwen3b_{tag}'
    final_head = f'{ckpt_dir}/head_final.safetensors'
    os.makedirs(ckpt_dir, exist_ok=True)
    if os.path.exists(final_head) and not FORCE_RETRAIN:
        print(f'skip trained {tag}: {final_head}')
    else:
        print(f'\n=== train {tag} spec={spec}')
        cmd = [
            sys.executable, 'colab/eagle5_train_pytorch.py',
            '--corpus-dir', CORPUS_DIR,
            '--frozen', FROZEN_OUT,
            '--ckpt-dir', ckpt_dir,
            '--epochs', str(TRAIN_EPOCHS),
            '--batch-size', str(TRAIN_BATCH),
            '--seq-len', '16',
            '--lr', str(spec['lr']),
            '--capture-layer', str(CAPTURE_LAYER),
            '--max-rows', str(TRAIN_MAX_ROWS),
            '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
            '--sparsity-head', 'off',
            '--seed', '0',
            '--calib-loss-weight', str(spec['calib_weight']),
            '--residual-delta-loss-weight', str(spec['residual_delta']),
            '--save-safetensors',
        ]
        if USE_TORCH_COMPILE and VRAM_GB >= 35:
            cmd.append('--compile')
        run(cmd)
    assert os.path.exists(final_head), f'training did not write {final_head}'
    trained.append((tag, spec, final_head))

if not trained:
    raise RuntimeError('No trained heads available.')

tau_results = {}
for tag, spec, ckpt in trained:
    out_path = f'{ARTIFACT_DIR}/eagle5_tau_{tag}.json'
    if os.path.exists(out_path) and not FORCE_TAU:
        print(f'skip tau {tag}: {out_path}')
    else:
        print(f'\n=== tau eval {tag}')
        run([
            sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
            '--ckpt', ckpt,
            '--frozen', FROZEN_OUT,
            '--corpus', CORPUS_DIR,
            '--out', out_path,
            '--depth', str(TAU_DEPTH),
            '--max-windows', str(EVAL_MAX_WINDOWS),
            '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
            '--base-tps', str(BASELINE_DEC_TPS),
            '--w4a8-multiplier', str(W4A8_MULTIPLIER),
            '--spec-efficiency', str(SPEC_EFFICIENCY),
        ])
    with open(out_path) as f:
        tau_results[tag] = json.load(f)
    tau_results[tag]['train_spec'] = spec
    tau_results[tag]['ckpt'] = ckpt

ranked_tau = sorted(
    tau_results.items(),
    key=lambda kv: (kv[1]['projection']['projected_dec_tps'], kv[1]['tau'], kv[1]['depth1_accept_rate']),
    reverse=True,
)
winner_tag, winner = ranked_tau[0]
tau_winner_path = f'{ARTIFACT_DIR}/eagle5_tau_winner.json'
with open(tau_winner_path, 'w') as f:
    json.dump({'winner': winner_tag, 'result': winner, 'all': tau_results}, f, indent=2, sort_keys=True)

print('\nTau race results:')
for tag, r in ranked_tau:
    print(f"  {tag:20s} tau={r['tau']:.3f} depth1={r['depth1_accept_rate']:.2%} full@{TAU_DEPTH}={r['full_accept_rate']:.2%} projected={r['projection']['projected_dec_tps']:.1f} spec={r['train_spec']}")
print(f'\nTau winner: {winner_tag} projected={winner["projection"]["projected_dec_tps"]:.1f} -> {tau_winner_path}')


In [ ]:
# Cell 4 - Frontier policy search + AWQ/W4A8 validation
if 'ranked_tau' not in globals():
    tau_winner_path = f'{ARTIFACT_DIR}/eagle5_tau_winner.json'
    assert os.path.exists(tau_winner_path), 'Run Cell 3 first, or provide eagle5_tau_winner.json.'
    tau_info = json.load(open(tau_winner_path))
    ranked_tau = sorted(
        tau_info['all'].items(),
        key=lambda kv: (kv[1]['projection']['projected_dec_tps'], kv[1]['tau'], kv[1]['depth1_accept_rate']),
        reverse=True,
    )

frontier_results = {}
frontier_candidates = ranked_tau[:FRONTIER_TOP_N]
for tag, r in frontier_candidates:
    out_path = f'{ARTIFACT_DIR}/eagle5_frontier_{tag}.json'
    if os.path.exists(out_path) and not FORCE_FRONTIER:
        print(f'skip frontier {tag}: {out_path}')
    else:
        print(f'\n=== frontier search {tag}')
        run([
            sys.executable, 'colab/eagle5_frontier_policy.py',
            '--ckpt', r['ckpt'],
            '--frozen', FROZEN_OUT,
            '--corpus', CORPUS_DIR,
            '--out', out_path,
            '--max-depth', str(FRONTIER_MAX_DEPTH),
            '--depths', '4,6,8,12',
            '--lattice-widths', '2,3,4',
            '--max-windows', str(EVAL_MAX_WINDOWS),
            '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
            '--base-tps', str(BASELINE_DEC_TPS),
            '--w4a8-multiplier', str(W4A8_MULTIPLIER),
            '--spec-efficiency', str(SPEC_EFFICIENCY),
        ])
    with open(out_path) as f:
        frontier_results[tag] = json.load(f)

frontier_ranked = sorted(
    frontier_results.items(),
    key=lambda kv: kv[1]['policies']['best_deployable']['projected_dec_tps'],
    reverse=True,
)
frontier_winner_tag, frontier_winner = frontier_ranked[0]
frontier_winner_path = f'{ARTIFACT_DIR}/eagle5_frontier_winner.json'
with open(frontier_winner_path, 'w') as f:
    json.dump({'winner': frontier_winner_tag, 'result': frontier_winner, 'all': frontier_results}, f, indent=2, sort_keys=True)

print('\nFrontier policy results:')
for tag, payload in frontier_ranked:
    best = payload['policies']['best_deployable']
    rd = payload['policies']['residual_delta_probe']
    print(f"  {tag:20s} deployable={best['kind']:34s} projected={best['projected_dec_tps']:.1f} accepted={best['accepted_draft_tokens_per_verify']:.2f} residual_cos={rd['cosine_mean']}")
print(f'\nFrontier winner: {frontier_winner_tag} -> {frontier_winner_path}')

VALIDATION_OUT = f'{ARTIFACT_DIR}/awq_w4a8_validation.json'
if os.path.exists(VALIDATION_OUT) and not FORCE_VALIDATE:
    print(f'skip AWQ validation: {VALIDATION_OUT}')
elif VRAM_GB < 20:
    print('Skipping W4A8 validation on <20 GB GPU; fake-quant validator needs fp16 Linears to instrument.')
else:
    run([
        sys.executable, 'colab/awq_w4a8_validate.py',
        '--model', MODEL_ID,
        '--smoothing', AWQ_OUT,
        '--out', VALIDATION_OUT,
        '--n-prompts', '30',
        '--max-new-tokens', '32',
        '--device', 'cuda',
    ])

if os.path.exists(VALIDATION_OUT):
    d = json.load(open(VALIDATION_OUT))
    print('\nAWQ validation:')
    print(f'  WITH AWQ:    KL={d["with_awq"]["mean_kl_per_token"]:.4f} match@32={d["with_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  WITHOUT AWQ: KL={d["without_awq"]["mean_kl_per_token"]:.4f} match@32={d["without_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  delta: KL={d["delta"]["kl_improvement"]:+.4f} match={d["delta"]["match_improvement"]:+.2%}')


In [ ]:
# Cell 5 - Summary and local bench command
tau_winner_path = f'{ARTIFACT_DIR}/eagle5_tau_winner.json'
frontier_winner_path = f'{ARTIFACT_DIR}/eagle5_frontier_winner.json'
validation_path = f'{ARTIFACT_DIR}/awq_w4a8_validation.json'
assert os.path.exists(tau_winner_path), 'Missing tau winner. Run Cell 3 first.'
assert os.path.exists(frontier_winner_path), 'Missing frontier winner. Run Cell 4 first.'

tau_info = json.load(open(tau_winner_path))
frontier_info = json.load(open(frontier_winner_path))
frontier_best = frontier_info['result']['policies']['best_deployable']
frontier_hints = frontier_info['result']['runtime_hints']
wtag = frontier_info['winner']

summary_lines = [
    '# Qwen-3B Full-Stack Pipeline Summary',
    '',
    f'Generated on `{GPU_NAME}` from `colab/qwen3b_full_stack.ipynb`.',
    '',
    '## Mode',
    '',
    f'- MAX_TPS_MODE: `{MAX_TPS_MODE}`',
    f'- corpus target: `{CORPUS_TARGET_SEQS}` sequences',
    f'- train rows: `{TRAIN_MAX_ROWS}`',
    f'- train epochs: `{TRAIN_EPOCHS}`',
    f'- frontier max depth: `{FRONTIER_MAX_DEPTH}`',
    '',
    '## Winners',
    '',
    f'- Tau winner: `{tau_info["winner"]}` projected `{tau_info["result"]["projection"]["projected_dec_tps"]:.1f}` dec_tps',
    f'- Frontier winner: `{wtag}`',
    f'- Best deployable policy: `{frontier_best["kind"]}`',
    f'- Accepted draft tokens / verify: `{frontier_best["accepted_draft_tokens_per_verify"]:.3f}`',
    f'- Projected deployable stack: `{frontier_best["projected_dec_tps"]:.1f}` dec_tps',
    '',
    '## Runtime Hints For Rust Wiring',
    '',
    '```json',
    json.dumps(frontier_hints, indent=2, sort_keys=True),
    '```',
    '',
    '## Artifacts',
    '',
]
for path in [FROZEN_OUT, AWQ_OUT, tau_winner_path, frontier_winner_path, validation_path]:
    if os.path.exists(path):
        summary_lines.append(f'- `{path}` ({os.path.getsize(path)/1e6:.1f} MB)')
if os.path.isdir(CORPUS_DIR):
    corpus_size = sum(os.path.getsize(os.path.join(r, fn)) for r, _, fns in os.walk(CORPUS_DIR) for fn in fns) / 1e9
    summary_lines.append(f'- `{CORPUS_DIR}/` ({corpus_size:.2f} GB)')

summary_lines.extend([
    '',
    '## What This Added',
    '',
    '- Variable-K Eagle policy search from the trained calibration head.',
    '- Entropy/margin routing to avoid wasting speculation on chaotic tokens.',
    '- Draft-lattice upper-bound probe for future tree verification.',
    '- Residual-delta-loss training variants to make draft states better multi-step simulators.',
    '',
    '## Local Bench Command',
    '',
    '```bash',
    f'EAGLE5_HEAD=artifacts/qwen3b_artifacts/checkpoints/eagle5_qwen3b_{wtag}/head_final.safetensors \\',
    '  TRIALS=10 TOKENS=128 \\',
    '  KERNEL_PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json \\',
    '  bash tools/bench/eagle5_paired_bench.sh',
    '```',
    '',
    'The Colab projection ranks candidates. The laptop paired bench is the real pass/fail for 100+ dec_tps.',
])
summary = '\n'.join(summary_lines) + '\n'
summary_path = f'{ARTIFACT_DIR}/summary.md'
with open(summary_path, 'w') as f:
    f.write(summary)
print(summary)
print(f'wrote {summary_path}')
